# Construct pydantic model from text input

In [27]:
from pydantic_ai import Agent 

agent = Agent(model="openrouter:nvidia/nemotron-3-super-120b-a12b:free")

result = await agent.run(" Give me an IT employee in sweden, keep it short")
result

AgentRunResult(output='**Name:** Erik Svensson  \n**Role:** IT Support Specialist  \n**Location:** Stockholm, Sweden  \n**Company:** TechConsult AB  \n**Skills:** Windows/Linux systems, network troubleshooting, user support, ITIL basics  \n**Language:** Swedish (native), English (fluent)')

In [28]:
print(result.output)

**Name:** Erik Svensson  
**Role:** IT Support Specialist  
**Location:** Stockholm, Sweden  
**Company:** TechConsult AB  
**Skills:** Windows/Linux systems, network troubleshooting, user support, ITIL basics  
**Language:** Swedish (native), English (fluent)


In [29]:
from pydantic import BaseModel, Field

class EmployeeModel(BaseModel):
    name: str
    age: int
    salary: int = Field(gt=25000, lt=150000)
    position: str 

result_1 = await agent.run(" Give me an IT employee in sweden, keep it short", output_type=EmployeeModel)

result_1

AgentRunResult(output=EmployeeModel(name='Anna Svensson', age=32, salary=75000, position='Software Engineer'))

In [30]:
print(result_1.output)

name='Anna Svensson' age=32 salary=75000 position='Software Engineer'


In [31]:
employee = result_1.output
employee

EmployeeModel(name='Anna Svensson', age=32, salary=75000, position='Software Engineer')

In [32]:
employee.name, employee.age, employee.salary, employee.position

('Anna Svensson', 32, 75000, 'Software Engineer')

In [33]:
employee.model_dump()

{'name': 'Anna Svensson',
 'age': 32,
 'salary': 75000,
 'position': 'Software Engineer'}

In [34]:
print(employee.model_dump_json(indent=2))

{
  "name": "Anna Svensson",
  "age": 32,
  "salary": 75000,
  "position": "Software Engineer"
}


several employees or a list of employees

In [35]:
result = await agent.run(
    """ Give me 3 employees in ai and dataengineering fields, roles can vary," \
    but salary must be consistent between 30000 and 500000""",
    output_type=list[EmployeeModel]
    )

employees = result.output
employees

[EmployeeModel(name='Alice Smith', age=29, salary=85000, position='AI Engineer'),
 EmployeeModel(name='Bob Lee', age=35, salary=120000, position='Data Engineering Lead'),
 EmployeeModel(name='Carol Nguyen', age=42, salary=95000, position='Machine Learning Specialist')]

In [36]:
len(employees)

3

In [39]:
for employe in employees:
    print(f"{employe.name} = and {employe.salary =}")

Alice Smith = and employe.salary =85000
Bob Lee = and employe.salary =120000
Carol Nguyen = and employe.salary =95000


## CV or resume model - a more complex and nested model

In [41]:
class ExperienceModel(BaseModel):
    title: str
    company: str
    description: str
    start_year: int
    end_year: int   

class EducationModel(BaseModel):
    title: str
    education_area: str
    school: str
    start_year: int
    end_year: int   

class CvModel(BaseModel):
    name: str
    age: int
    experience: list[ExperienceModel]
    education: list[EducationModel]

result = await agent.run(
    "create a swedish person applying for a data engineering",
    output_type=CvModel
)

resume = result.output
resume

CvModel(name='Erik Johansson', age=28, experience=[ExperienceModel(title='Junior Data Engineer', company='Spotify', description='Developed ETL pipelines using Python and Apache Spark, optimized data warehouse performance, collaborated with data scientists to implement machine learning models in production.', start_year=2022, end_year=2024), ExperienceModel(title='Data Analysis Intern', company='Ericsson', description='Assisted in building data dashboards with Tableau, performed SQL queries for business insights, supported machine learning model deployment, and maintained data pipelines.', start_year=2021, end_year=2022)], education=[EducationModel(title='Master of Science in Computer Science', education_area='Computer Science', school='KTH Royal Institute of Technology', start_year=2020, end_year=2022), EducationModel(title='Bachelor of Science in Information Technology', education_area='Information Technology', school='Lund University', start_year=2017, end_year=2020)])

In [45]:
resume.name, resume.age

('Erik Johansson', 28)

In [49]:
resume.experience[1].title

'Data Analysis Intern'

In [52]:
resume.model_dump().keys()

dict_keys(['name', 'age', 'experience', 'education'])

## Optional postprocessing -> load into duckdb and unnest

In [94]:
import dlt

pipeline = dlt.pipeline(
    pipeline_name="resume_json_duckdb",
    destination=dlt.destinations.duckdb("cv_duckdb"),
    dataset_name="staging"
)

info = pipeline.run(data=[resume.model_dump()], loader_file_format="jsonl", table_name="cv_entries")
print(info)

Pipeline resume_json_duckdb load step completed in 0.14 seconds
1 load package(s) were loaded to destination duckdb and into dataset staging
The duckdb destination used duckdb:///c:\Users\linus\Documents\github\llmops_linus_larsson_mlo25\04_pydanticai_structured_outputs\cv_duckdb location to store data
Load package 1776086507.9347703 is LOADED and contains no failed jobs


In [100]:
import duckdb

with duckdb.connect("cv_duckdb") as conn:
    desc = conn.sql("desc").df()
    cv_entries = conn.sql("from staging.cv_entries").df()
    educations = conn.sql("from staging.cv_entries__education").df()
    experiences = conn.sql("from staging.cv_entries__experience").df()

desc

,database,schema,name,column_names,column_types,temporary
0,cv_duckdb,staging,_dlt_loads,"[load_id, schema_name, status, inserted_at, sc...","[VARCHAR, VARCHAR, BIGINT, TIMESTAMP WITH TIME...",False
1,cv_duckdb,staging,_dlt_pipeline_state,"[version, engine_version, pipeline_name, state...","[BIGINT, BIGINT, VARCHAR, VARCHAR, TIMESTAMP W...",False
2,cv_duckdb,staging,_dlt_version,"[version, engine_version, inserted_at, schema_...","[BIGINT, BIGINT, TIMESTAMP WITH TIME ZONE, VAR...",False
3,cv_duckdb,staging,cv_entries,"[name, age, _dlt_load_id, _dlt_id]","[VARCHAR, BIGINT, VARCHAR, VARCHAR]",False
4,cv_duckdb,staging,cv_entries__education,"[title, education_area, school, start_year, en...","[VARCHAR, VARCHAR, VARCHAR, BIGINT, BIGINT, VA...",False
5,cv_duckdb,staging,cv_entries__experience,"[title, company, description, start_year, end_...","[VARCHAR, VARCHAR, VARCHAR, BIGINT, BIGINT, VA...",False


In [96]:
cv_entries

,name,age,_dlt_load_id,_dlt_id
0,Erik Johansson,28,1776085171.317044,dC5eVJK21IUAXg
1,Erik Johansson,28,1776086507.9347703,io5HtV4FBxFLvw


In [101]:
educations

,title,education_area,school,start_year,end_year,_dlt_parent_id,_dlt_list_idx,_dlt_id
0,Master of Science in Computer Science,Computer Science,KTH Royal Institute of Technology,2020,2022,dC5eVJK21IUAXg,0,y4fBvIOTngjXuQ
1,Bachelor of Science in Information Technology,Information Technology,Lund University,2017,2020,dC5eVJK21IUAXg,1,gQGuUo1RimDWvw
2,Master of Science in Computer Science,Computer Science,KTH Royal Institute of Technology,2020,2022,io5HtV4FBxFLvw,0,/20lhFJ96KmieQ
3,Bachelor of Science in Information Technology,Information Technology,Lund University,2017,2020,io5HtV4FBxFLvw,1,J3CPOHh9VPAJsg


In [87]:
experiences

,title,company,description,start_year,end_year,_dlt_parent_id,_dlt_list_idx,_dlt_id
0,Junior Data Engineer,Spotify,Developed ETL pipelines using Python and Apach...,2022,2024,dC5eVJK21IUAXg,0,uDagUm63hfqCcA
1,Data Analysis Intern,Ericsson,Assisted in building data dashboards with Tabl...,2021,2022,dC5eVJK21IUAXg,1,qQJ2ajbSOWXBFg


In [104]:
with duckdb.connect("cv_duckdb") as conn:
    df = conn.sql("""
        SELECT
            cv.name,
            cv.age,
            ex.company,
            ex.description AS experience_description,
            ex.start_year AS experience_start_year,
            ex.end_year AS experience_end_year,
            e.title,
            e.education_area,
            e.school,
            e.start_year AS education_start_year,
            e.end_year AS education_end_year
        FROM staging.cv_entries AS cv
        LEFT JOIN staging.cv_entries__education AS e
            ON cv._dlt_id = e._dlt_parent_id
        LEFT JOIN staging.cv_entries__experience AS ex
            ON cv._dlt_id = ex._dlt_parent_id
    """).df()

df

,name,age,company,experience_description,experience_start_year,experience_end_year,title,education_area,school,education_start_year,education_end_year
0,Erik Johansson,28,Ericsson,Assisted in building data dashboards with Tabl...,2021,2022,Master of Science in Computer Science,Computer Science,KTH Royal Institute of Technology,2020,2022
1,Erik Johansson,28,Ericsson,Assisted in building data dashboards with Tabl...,2021,2022,Bachelor of Science in Information Technology,Information Technology,Lund University,2017,2020
2,Erik Johansson,28,Ericsson,Assisted in building data dashboards with Tabl...,2021,2022,Master of Science in Computer Science,Computer Science,KTH Royal Institute of Technology,2020,2022
3,Erik Johansson,28,Ericsson,Assisted in building data dashboards with Tabl...,2021,2022,Bachelor of Science in Information Technology,Information Technology,Lund University,2017,2020
4,Erik Johansson,28,Spotify,Developed ETL pipelines using Python and Apach...,2022,2024,Master of Science in Computer Science,Computer Science,KTH Royal Institute of Technology,2020,2022
5,Erik Johansson,28,Spotify,Developed ETL pipelines using Python and Apach...,2022,2024,Bachelor of Science in Information Technology,Information Technology,Lund University,2017,2020
6,Erik Johansson,28,Spotify,Developed ETL pipelines using Python and Apach...,2022,2024,Master of Science in Computer Science,Computer Science,KTH Royal Institute of Technology,2020,2022
7,Erik Johansson,28,Spotify,Developed ETL pipelines using Python and Apach...,2022,2024,Bachelor of Science in Information Technology,Information Technology,Lund University,2017,2020
